# Compare optimised pipeline runs (CSV diagnostics)

This notebook helps you compare multiple `optimised_processing` runs by:
- locating one-row diagnostics CSVs (recursively)
- concatenating them into a single table
- computing a few derived metrics (e.g. clearing %)
- plotting comparisons across `run_tag` / settings

The pipeline writes these diagnostics from the legacy method:
- `*_summary.csv` (one row per run)
- `*_sr_scale_verify.csv` (one row per run; band percentile audit)

If nothing is found, update `SEARCH_ROOTS` in Cell 2 to point at your run work directory.

# Terminal run prompts (EASI)

Copy/paste reference for running the NDVI build and three EDS A/B variants.

## Common variables (set once)

```bash
export EASI_REPO="/home/jovyan/work-easi-eds"
export EDS_BUCKET="dcceew-eds-data"
export EDS_PREFIX="AROAZ6PFZYT4B4C7MNRHV:robotmcgregor/eds"

export NDVI_WORK="/home/jovyan/scratch/eds-work-optimised"
export EDS_WORK="/home/jovyan/scratch/eds-work-processing"

cd "$EASI_REPO"
```

## 1) Build NDVI products (required once per tile/date)

```bash
python "$EASI_REPO/scripts/easi-scripts/optimised_ndvi/scripts/ndvi_master_pipeline.py" \
  --tile p089r080 \
  --s3-bucket "$EDS_BUCKET" \
  --s3-prefix "$EDS_PREFIX" \
  --work-dir "$NDVI_WORK" \
  --cloud-max 40 \
  --start-date 2025-06-07 \
  --end-date 2026-01-09
```

## 2) Run EDS optimised processing (3 A/B variants)

All variants below write run-scoped outputs via `--run-id` and emit diagnostics CSVs via `--diagnostics`.

### Variant A: FORCE no SR scaling (debug baseline)

```bash
python "$EASI_REPO/scripts/easi-scripts/optimised_processing/scripts/eds_master_pipeline_optimised.py" \
  --tile p089r080 \
  --start-date 2025-06-07 \
  --end-date 2026-01-09 \
  --s3-bucket "$EDS_BUCKET" \
  --s3-prefix "$EDS_PREFIX" \
  --work-dir "$EDS_WORK" \
  --cloud-max 40 \
  --lookback 10 \
  --copy-to-home \
  --verbose \
  --diagnostics \
  --dlj-troubleshoot \
  --legacy-no-auto-sr-scale \
  --run-id run-no-auto-sr-scale
```

### Variant B: AUTO SR scaling (recommended default)

```bash
python "$EASI_REPO/scripts/easi-scripts/optimised_processing/scripts/eds_master_pipeline_optimised.py" \
  --tile p089r080 \
  --start-date 2025-06-07 \
  --end-date 2026-01-09 \
  --s3-bucket "$EDS_BUCKET" \
  --s3-prefix "$EDS_PREFIX" \
  --work-dir "$EDS_WORK" \
  --cloud-max 40 \
  --lookback 10 \
  --copy-to-home \
  --verbose \
  --diagnostics \
  --dlj-troubleshoot \
  --run-id run-auto-scale
```

### Variant C: MANUAL SR scaling (force 10000)

```bash
python "$EASI_REPO/scripts/easi-scripts/optimised_processing/scripts/eds_master_pipeline_optimised.py" \
  --tile p089r080 \
  --start-date 2025-06-07 \
  --end-date 2026-01-09 \
  --s3-bucket "$EDS_BUCKET" \
  --s3-prefix "$EDS_PREFIX" \
  --work-dir "$EDS_WORK" \
  --cloud-max 40 \
  --lookback 10 \
  --copy-to-home \
  --verbose \
  --diagnostics \
  --dlj-troubleshoot \
  --legacy-sr-scale 10000 \
  --run-id run-forced-10000
```

Tip: you generally don’t need `--rebase` for A/B runs if you keep unique `--run-id` values.

In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
import os
import re

import pandas as pd

# --- Configure where to search ---
# This notebook is intended to be run on EASI (Linux/Jupyter).
# The diagnostics CSVs are written to a local run folder like:
#   <work-dir>/<tile>/<run_tag>/diagnostics/*_summary.csv
#
# If you don't know the exact work-dir, start by searching common locations.

def default_search_roots() -> list[Path]:
    candidates: list[Path] = []
    # EASI/Jupyter common locations
    for p in (
        Path.cwd().resolve(),
        Path('/home/jovyan'),
        Path('/home/jovyan/work'),
        Path('/home/jovyan/shared'),
        Path('/mnt'),
        Path('/data'),
    ):
        try:
            if p.exists():
                candidates.append(p)
        except Exception:
            pass

    # De-dupe, preserve order
    out: list[Path] = []
    seen: set[str] = set()
    for p in candidates:
        sp = str(p)
        if sp not in seen:
            out.append(p)
            seen.add(sp)
    return out


SEARCH_ROOTS = default_search_roots()

# File patterns we expect from the legacy diagnostics step
SUMMARY_GLOB = '**/*_summary.csv'
SR_VERIFY_GLOB = '**/*_sr_scale_verify.csv'

print('CWD:', Path.cwd().resolve())
print('Search roots:')
for r in SEARCH_ROOTS:
    print(' -', r)

In [ ]:
@dataclass(frozen=True)
class LocatedFile:
    path: Path
    run_tag: str | None
    tile_from_path: str | None
    diag_suffix: str | None


def _infer_run_tag_and_tile_from_path(p: Path) -> tuple[str | None, str | None]:
    # Typical layout: <work-dir>/<tile>/<run_tag>/diagnostics/<file.csv>
    parts = list(p.parts)
    try:
        i = parts.index('diagnostics')
    except ValueError:
        return None, None
    if i - 1 < 0:
        return None, None
    run_tag = parts[i - 1]
    tile = parts[i - 2] if i - 2 >= 0 else None
    # sanity check tile like p089r080
    if tile and not re.match(r'^p\d{3}r\d{3}$', tile, flags=re.IGNORECASE):
        tile = None
    return run_tag, tile.lower() if tile else None


def _infer_diag_suffix_from_filename(p: Path) -> str | None:
    # The legacy script writes names like:
    #   <diag_name_base>_<diag_suffix>_summary.csv
    # where diag_suffix is like:
    #   sr-auto-10000_base-nodataaware
    #   sr-manual-10000_base-legacy
    name = p.name
    if name.endswith('_summary.csv'):
        core = name[:-len('_summary.csv')]
    elif name.endswith('_sr_scale_verify.csv'):
        core = name[:-len('_sr_scale_verify.csv')]
    else:
        core = p.stem

    # Heuristic: suffix starts after last occurrence of '_vi-'
    # Example: ..._vi-ndvi_e32756_<suffix>
    m = re.search(r'(_vi-[^_]+_e\d+)_', core)
    if not m:
        return None
    return core[m.end():] or None


def find_csvs(roots: list[Path], pattern: str) -> list[LocatedFile]:
    found: list[LocatedFile] = []
    for root in roots:
        root = Path(root)
        if not root.exists():
            continue
        for p in root.glob(pattern):
            if not p.is_file():
                continue
            run_tag, tile_from_path = _infer_run_tag_and_tile_from_path(p)
            diag_suffix = _infer_diag_suffix_from_filename(p)
            found.append(LocatedFile(path=p, run_tag=run_tag, tile_from_path=tile_from_path, diag_suffix=diag_suffix))
    # de-dupe, stable sort
    uniq = {str(x.path): x for x in found}
    return sorted(uniq.values(), key=lambda x: str(x.path))


def read_one_row_csvs(files: list[LocatedFile]) -> pd.DataFrame:
    rows: list[pd.DataFrame] = []
    for lf in files:
        try:
            df = pd.read_csv(lf.path)
        except Exception as e:
            print('[WARN] failed to read:', lf.path, '->', e)
            continue
        if df.shape[0] != 1:
            print('[WARN] expected 1 row but got', df.shape[0], 'for', lf.path)
        df = df.copy()
        df['source_path'] = str(lf.path)
        df['run_tag'] = lf.run_tag
        df['tile_from_path'] = lf.tile_from_path
        df['diag_suffix'] = lf.diag_suffix
        rows.append(df)
    if not rows:
        return pd.DataFrame()
    out = pd.concat(rows, ignore_index=True)
    # prefer tile from CSV, fall back to path
    if 'tile' not in out.columns:
        out['tile'] = out['tile_from_path']
    else:
        out['tile'] = out['tile'].fillna(out['tile_from_path'])
    return out

In [ ]:
# Locate and load summary CSVs
summary_files = find_csvs(SEARCH_ROOTS, SUMMARY_GLOB)
print('Found summary CSVs:', len(summary_files))
if summary_files:
    print('First few:')
    for lf in summary_files[:5]:
        print(' -', lf.path)

df_summary = read_one_row_csvs(summary_files)
print('Summary table shape:', df_summary.shape)
df_summary.head(5)

In [ ]:
# If we found nothing, show a hint and stop early
if df_summary.empty:
    raise RuntimeError(
        'No *_summary.csv files found under SEARCH_ROOTS.\n'
        'On EASI, set SEARCH_ROOTS in Cell 2 to the pipeline work directory (the folder that contains <tile>/<run_tag>/diagnostics/).\n'
        'Example: SEARCH_ROOTS = [Path("/home/jovyan/<somewhere>")]'
    )

In [ ]:
# Derive comparison-friendly columns (clearing %, etc.)
import numpy as np
import matplotlib.pyplot as plt

def _sum_cols(df: pd.DataFrame, cols: list[str]) -> pd.Series:
    existing = [c for c in cols if c in df.columns]
    if not existing:
        return pd.Series([pd.NA] * len(df), index=df.index)
    return df[existing].fillna(0).sum(axis=1)

df = df_summary.copy()

# These are the main clearing classes used by the legacy method
clearing_classes = [34, 35, 36, 37, 38, 39]
clearing_cols = [f'dll_count_{c}' for c in clearing_classes]

df['dll_clearing_px'] = _sum_cols(df, clearing_cols)
df['dll_ndvi_only_px'] = df['dll_count_3'] if 'dll_count_3' in df.columns else pd.NA
df['dll_no_clearing_px'] = df['dll_count_10'] if 'dll_count_10' in df.columns else pd.NA

if 'dll_total_px' in df.columns:
    df['dll_clearing_pct'] = (df['dll_clearing_px'] / df['dll_total_px']) * 100.0
else:
    df['dll_clearing_pct'] = pd.NA

# Helpful label for charts
df['run_label'] = (df.get('run_tag', pd.Series([''] * len(df))).fillna('') + ' | ' + df.get('diag_suffix', pd.Series([''] * len(df))).fillna('')).str.strip(' |')

# Sort for readability
sort_cols = [c for c in ['tile', 'end_date', 'run_tag', 'sr_scale_source', 'sr_scale_factor', 'baseline_include_nodata'] if c in df.columns]
if sort_cols:
    df = df.sort_values(sort_cols)

display_cols = [c for c in [
    'tile', 'run_tag', 'diag_suffix', 'start_date', 'end_date',
    'sr_scale_source', 'sr_scale_factor', 'baseline_include_nodata',
    'dll_total_px', 'dll_clearing_px', 'dll_clearing_pct',
    'ndviDiffStdErr_pct_ge_2_5', 'ndviDiffStdErr_pct_ge_6_0', 'ndviDiffStdErr_pct_ge_10_0',
    'source_path',
] if c in df.columns]

df[display_cols].head(25)

In [ ]:
# Plot 1: clearing % by run (one chart per tile)
def plot_clearing_by_tile(df: pd.DataFrame) -> None:
    if 'dll_clearing_pct' not in df.columns:
        print('[WARN] dll_clearing_pct not present; cannot plot.')
        return

    tiles = [t for t in sorted(df['tile'].dropna().unique())]
    if not tiles:
        print('[WARN] No tiles found to plot.')
        return

    for tile in tiles:
        sub = df[df['tile'] == tile].copy()
        sub = sub.dropna(subset=['dll_clearing_pct'])
        if sub.empty:
            continue
        sub = sub.sort_values(['run_tag', 'diag_suffix'], na_position='last')

        plt.figure(figsize=(10, max(3, 0.35 * len(sub))))
        y = np.arange(len(sub))
        plt.barh(y, sub['dll_clearing_pct'].astype(float))
        plt.yticks(y, sub['run_label'])
        plt.xlabel('DLL clearing pixels (% of total)')
        plt.title(f'{tile}: clearing % by run')
        plt.grid(axis='x', alpha=0.2)
        plt.tight_layout()
        plt.show()

plot_clearing_by_tile(df)

In [ ]:
# Plot 2: clearing pixels vs SR scaling factor (quick correlation check)
if 'dll_clearing_px' in df.columns and 'sr_scale_factor' in df.columns:
    plt.figure(figsize=(8, 4))
    plt.scatter(df['sr_scale_factor'].astype(float), df['dll_clearing_px'].astype(float), alpha=0.7)
    plt.xlabel('sr_scale_factor')
    plt.ylabel('dll_clearing_px (sum of classes 34..39)')
    plt.title('Clearing pixels vs SR scaling factor')
    plt.grid(alpha=0.2)
    plt.tight_layout()
    plt.show()
else:
    print('[INFO] Missing columns for SR scale scatter plot')

In [ ]:
# Load SR scaling verification CSVs (optional)
sr_files = find_csvs(SEARCH_ROOTS, SR_VERIFY_GLOB)
print('Found SR verify CSVs:', len(sr_files))
if sr_files:
    print('First few:')
    for lf in sr_files[:5]:
        print(' -', lf.path)

df_sr = read_one_row_csvs(sr_files)
print('SR verify table shape:', df_sr.shape)
df_sr.head(3)

In [ ]:
# SR scaling audit plot: raw vs scaled median (p50) for start/end bands
def _maybe_col(df: pd.DataFrame, col: str) -> bool:
    return col in df.columns and df[col].notna().any()

if df_sr.empty:
    print('[INFO] No sr_scale_verify CSVs found (this is optional).')
else:
    dfp = df_sr.copy()
    dfp['run_label'] = (dfp.get('run_tag', pd.Series([''] * len(dfp))).fillna('') + ' | ' + dfp.get('diag_suffix', pd.Series([''] * len(dfp))).fillna('')).str.strip(' |')
    # Expect columns like: start_b2_raw_p50 / start_b2_scaled_p50
    targets = []
    for which in ['start', 'end']:
        for band in ['b2', 'b3', 'b5', 'b6']:
            raw = f'{which}_{band}_raw_p50'
            scaled = f'{which}_{band}_scaled_p50'
            if _maybe_col(dfp, raw) and _maybe_col(dfp, scaled):
                targets.append((which, band, raw, scaled))

    if not targets:
        print('[INFO] No p50 columns found to plot in SR verify table.')
    else:
        for which, band, raw_col, scaled_col in targets:
            plt.figure(figsize=(10, 4))
            x = np.arange(len(dfp))
            plt.plot(x, dfp[raw_col].astype(float), label='raw (reconstructed)')
            plt.plot(x, dfp[scaled_col].astype(float), label='scaled (used by method)')
            plt.xticks(x, dfp['run_label'], rotation=45, ha='right')
            plt.ylabel('median (p50)')
            plt.title(f'SR scaling audit: {which} {band} p50')
            plt.grid(alpha=0.2)
            plt.legend()
            plt.tight_layout()
            plt.show()

In [ ]:
# Save concatenated tables for later use
out_dir = Path.cwd().resolve() / 'outputs' / 'diagnostics_concat'
out_dir.mkdir(parents=True, exist_ok=True)

summary_out = out_dir / 'all_runs_summary_concat.csv'
df.to_csv(summary_out, index=False)
print('Wrote:', summary_out)

if not df_sr.empty:
    sr_out = out_dir / 'all_runs_sr_scale_verify_concat.csv'
    df_sr.to_csv(sr_out, index=False)
    print('Wrote:', sr_out)